# M48 — Random Forest Cross-Validation

This notebook evaluates the existing AuthentiScore Random Forest using **5-fold stratified cross-validation** on the training split created in M6.

The untouched M6 test split is kept separate for the final held-out evaluation. This keeps the cross-validation step consistent with the existing training workflow.


In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
TRAIN_DIR = ROOT / "data" / "processed" / "train_test"
RESULTS_DIR = ROOT / "data" / "processed"

X_train = pd.read_csv(TRAIN_DIR / "X_train.csv")
y_train = pd.read_csv(TRAIN_DIR / "y_train.csv").squeeze()

X_test = pd.read_csv(TRAIN_DIR / "X_test.csv")
y_test = pd.read_csv(TRAIN_DIR / "y_test.csv").squeeze()

print("Training shape:", X_train.shape)
print("Test shape    :", X_test.shape)
print("Training target distribution:")
print(y_train.value_counts())


Training shape: (556, 9)
Test shape    : (140, 9)
Training target distribution:
fake
0    278
1    278
Name: count, dtype: int64


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print("✅ 5-fold cross-validation completed.")


✅ 5-fold cross-validation completed.


In [3]:
metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

summary = []

for metric in metrics:
    scores = cv_results[f"test_{metric}"]

    summary.append({
        "Metric": metric.upper().replace("_", "-"),
        "Mean": scores.mean(),
        "Std": scores.std(),
        "Min": scores.min(),
        "Max": scores.max()
    })

summary_df = pd.DataFrame(summary)

print("Cross-Validation Results")
print("=" * 60)
display(summary_df.round(4))


Cross-Validation Results


,Metric,Mean,Std,Min,Max
0,ACCURACY,0.9460,0.0190,0.9279,0.9732
1,PRECISION,0.9502,0.0299,0.9123,1.0000
2,RECALL,0.9426,0.0285,0.8929,0.9818
3,F1,0.9459,0.0190,0.9259,0.9725
4,ROC-AUC,0.9919,0.0033,0.9875,0.9966


In [4]:
fold_results = pd.DataFrame({
    "fold": range(1, 6),
    "accuracy": cv_results["test_accuracy"],
    "precision": cv_results["test_precision"],
    "recall": cv_results["test_recall"],
    "f1": cv_results["test_f1"],
    "roc_auc": cv_results["test_roc_auc"]
})

print("Per-fold results")
display(fold_results.round(4))


Per-fold results


,fold,accuracy,precision,recall,f1,roc_auc
0,1,0.9732,1.0000,0.9464,0.9725,0.9946
1,2,0.9279,0.9123,0.9455,0.9286,0.9906
2,3,0.9640,0.9474,0.9818,0.9643,0.9966
3,4,0.9279,0.9615,0.8929,0.9259,0.9875
4,5,0.9369,0.9298,0.9464,0.9381,0.9903


In [5]:
summary_path = RESULTS_DIR / "cross_validation_results.csv"
fold_path = RESULTS_DIR / "cross_validation_fold_results.csv"

summary_df.round(6).to_csv(summary_path, index=False)
fold_results.round(6).to_csv(fold_path, index=False)

print(f"✅ Summary saved to: {summary_path}")
print(f"✅ Fold results saved to: {fold_path}")


✅ Summary saved to: c:\Users\Acer\Documents\AuthentiScore\data\processed\cross_validation_results.csv
✅ Fold results saved to: c:\Users\Acer\Documents\AuthentiScore\data\processed\cross_validation_fold_results.csv


## M48 Interpretation

- **Mean** = average performance across the 5 validation folds.
- **Std** = variation across folds; lower values indicate more consistent fold-level performance.
- The M6 test split remains untouched by this cross-validation step and can be used later for the final held-out evaluation.
- The Random Forest configuration matches M8: `n_estimators=300`, `random_state=42`, `n_jobs=-1`.
